# Individual Search

> **Source:** `repo1/multi_agent_research_system.py` → `demo_individual_search()`


## Imports


In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.types import Send
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, BaseMessage
from typing_extensions import TypedDict, Annotated
from typing import Literal
from pydantic import BaseModel, Field
import operator
import json
from dotenv import load_dotenv


## Setup


In [ ]:
load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

creative_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)


## Class: `SearchTaskState`


In [ ]:
class SearchTaskState(TypedDict):
    search_query: str
    findings: Annotated[list[dict], operator.add]


## Helper: `search_agent`


In [ ]:
def search_agent(state: SearchTaskState) -> dict:
    """
    Executes one search query and returns findings.
    Each instance runs in parallel via the Send API.
    """
    query = state["search_query"]

    response = llm.invoke(
        [
            SystemMessage(
                content=(
                    "You are a web research agent. For the given search query, "
                    "provide 2-3 key findings. Each finding should have a 'title' "
                    "and 'detail' field. Return a JSON array. No markdown."
                )
            ),
            HumanMessage(content=f"Search query: {query}"),
        ]
    )

    try:
        results = json.loads(response.content)
    except json.JSONDecodeError:
        results = [{"title": query, "detail": response.content}]

    # Tag each finding with the query it came from
    for r in results:
        r["source_query"] = query

    return {"findings": results}


## Demo: Individual Search


In [ ]:
def demo_individual_search():
    """Demo just the search agent for testing."""

    print("Individual Search Agent Test:\n")

    # Test the search agent directly
    result = search_agent(
        {"search_query": "LangGraph multi-agent patterns", "findings": []}
    )

    print(f"Findings from search:")
    for f in result["findings"]:
        print(f"  - {f.get('title', 'N/A')}: {f.get('detail', 'N/A')[:80]}...")


## Run


In [ ]:
demo_individual_search()
